In [7]:
import sys
from pathlib import Path

# Fix path for Windows
sys.path.insert(0, r'C:\Users\LENOVO\quant\src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import MarketDataLoader
from src.data.preprocessor import DataPreprocessor
from risk.metrics import RiskMetrics
from utils.visualization import FinancialVisualizer

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ All modules loaded successfully")

✓ All modules loaded successfully


In [8]:
loader = MarketDataLoader(data_dir=r'C:\Users\LENOVO\quant\data\raw')
ticker = 'AAPL'
data = loader.fetch_data(tickers=ticker, start_date='2020-01-01', end_date='2023-12-31')

print(f"\nData shape: {data.shape}")
print(f"Columns: {data.columns.tolist()}")
print("\nFirst 5 rows:")
data.head()

Fetching data for ['AAPL'] from 2020-01-01 to 2023-12-31...
Data saved to C:\Users\LENOVO\quant\data\raw\AAPL_2020-01-01_2023-12-31.csv
✓ Fetched 1006 rows of data

Data shape: (1006, 5)
Columns: [('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]

First 5 rows:


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-01-02,72.468269,72.528589,71.223267,71.476607,135480400
2020-01-03,71.763710,72.523738,71.539322,71.696152,146322800
2020-01-06,72.335571,72.374177,70.634554,70.885487,118387200
2020-01-07,71.995384,72.600991,71.775819,72.345235,108872000
2020-01-08,73.153496,73.455095,71.698581,71.698581,132079200


In [9]:
preprocessor = DataPreprocessor()
print("=== Missing Data Analysis ===")
missing_info = preprocessor.check_missing_data(data)
if len(missing_info) > 0:
    display(missing_info)
else:
    print("✓ No missing data found")

print("\n=== Data Quality Validation ===")
quality_report = preprocessor.validate_data_quality(data)
print(f"Valid: {quality_report['is_valid']}")
if quality_report['issues']:
    for issue in quality_report['issues']:
        print(f"  - {issue}")
else:
    print("✓ No issues found")

=== Missing Data Analysis ===
✓ No missing data found

=== Data Quality Validation ===
Valid: True
✓ No issues found


In [11]:
# Handle both single-ticker and multi-ticker data formats
print(f"Data columns: {data.columns.tolist()[:5]}...")  # Show first 5 columns

# Check if data has MultiIndex columns
if isinstance(data.columns, pd.MultiIndex):
    # Multi-ticker format: data[('AAPL', 'Adj Close')]
    prices = data[(ticker, 'Adj Close')]
    print(f"✓ Extracted prices from multi-ticker format")
else:
    # Single-ticker format: data['Adj Close']
    prices = data['Adj Close']
    print(f"✓ Extracted prices from single-ticker format")

print(f"\nPrice data shape: {prices.shape}")
print(f"First 5 prices:\n{prices.head()}")

# Now create RiskMetrics and calculate returns
rm = RiskMetrics()
simple_returns = rm.simple_returns(prices)
log_returns = rm.log_returns(prices)

print("\n=== Returns Comparison (First 10 Days) ===")
comparison = pd.DataFrame({
    'Simple Returns': simple_returns,
    'Log Returns': log_returns,
    'Difference': simple_returns - log_returns
})
print(comparison.head(10))

Data columns: [('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]...


KeyError: ('AAPL', 'Adj Close')

In [12]:
vol_21d = rm.rolling_volatility(log_returns, window=21)
vol_63d = rm.rolling_volatility(log_returns, window=63)
realized_vol = rm.realized_volatility(log_returns)

print("=== Volatility Analysis ===")
print(f"Realized Volatility (2020-2023): {realized_vol:.4f} or {100*realized_vol:.2f}%")
print(f"\nCurrent 21-day volatility: {vol_21d.iloc[-1]:.4f}")
print(f"Current 63-day volatility: {vol_63d.iloc[-1]:.4f}")

NameError: name 'rm' is not defined

In [2]:
summary = rm.summary_statistics(prices, log_returns)
print("=== Complete Summary Statistics ===")
display(summary)

sharpe = summary.loc['Sharpe Ratio', 'Value']
print(f"\n=== Sharpe Ratio: {sharpe:.3f} ===")

NameError: name 'rm' is not defined

In [7]:
viz = FinancialVisualizer()
fig = viz.plot_comprehensive_analysis(prices, log_returns, ticker=ticker)
plt.show()

NameError: name 'prices' is not defined

In [8]:
fig = viz.plot_volatility(log_returns, vol_windows=[21, 63, 252], title=f'{ticker} - Rolling Volatility')
plt.show()

NameError: name 'log_returns' is not defined

In [9]:
fig = viz.plot_drawdown(prices, title=f'{ticker} Drawdown Analysis')
plt.show()

NameError: name 'prices' is not defined

In [10]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
multi_data = loader.fetch_data(tickers, '2020-01-01', '2023-12-31')

summaries = {}
for t in tickers:
    prices_i = multi_data[t]['Adj Close']
    returns_i = rm.log_returns(prices_i)
    summaries[t] = {
        'Total Return (%)': 100 * (prices_i.iloc[-1] / prices_i.iloc[0] - 1),
        'Annual Return (%)': 100 * returns_i.mean() * 252,
        'Annual Vol (%)': 100 * rm.realized_volatility(returns_i),
        'Sharpe Ratio': rm.sharpe_ratio(returns_i),
        'Max DD (%)': 100 * rm.max_drawdown(prices_i)
    }

comparison_df = pd.DataFrame(summaries).T
print("=== Tech Stock Comparison (2020-2023) ===")
display(comparison_df.round(2))

Fetching data for ['AAPL', 'MSFT', 'GOOGL', 'AMZN'] from 2020-01-01 to 2023-12-31...
Data saved to ..\data\raw\AAPL_MSFT_GOOGL_AMZN_2020-01-01_2023-12-31.csv
✓ Fetched 1006 rows of data


KeyError: 'Adj Close'